# Time Series Analysis in Medicine and Biology
## Practical Course — University of Tübingen · PfeiferLab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamsaraE/time-series-medicine-biology/blob/main/final-projects/02_rna_gam_model.ipynb)

---

# Final Project 02 — RNA Seasonal Analysis with Cyclic-Spline GAMs (IR vs IS)

This project builds directly on the **RNA layer** of the multi-omics dataset from Final Project 01.
It describes the **modelling workflow**; you implement it.

**Goal:**
1. Test whether RNA expression has an **annual cyclic** (seasonal) structure.
2. Test whether that seasonal modulation **differs between IR and IS** immune states.

The approach is **gene-level** modelling with **cyclic cubic-spline GAMs**, comparing three nested
models (M0: subject baseline only, M1: common seasonality, M2: group-specific seasonality) by AIC.

**Data — from the course page (not GitHub):** `RNA_df_Data.csv` + `RNA_annotation_colData.csv`,
the same files as Final Project 01. The annotation provides `SubjectID`, `Date`, and `IRIS`.

**What you submit:** a clean notebook with run cells, the genome-wide seasonality scan, top-gene
seasonal-curve plots (IR vs. IS), and a short interpretation of whether RNA seasonality is global
or sparse, and whether it differs by group.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.gam.api import GLMGam
from statsmodels.gam.smooth_basis import CyclicCubicSplines
from pathlib import Path

DATA_DIR = Path("data/multiomics")   # folder with the course-page files (see Final Project 01)

rna  = pd.read_csv(DATA_DIR / "RNA_df_Data.csv")              # samples x genes
anno = pd.read_csv(DATA_DIR / "RNA_annotation_colData.csv")   # same rows, same order
anno["Date"] = pd.to_datetime(anno["Date"], format="%y-%m-%d %H:%M:%S", errors="coerce")

# Build day-of-year for the cyclic spline (annual period)
anno["doy"] = anno["Date"].dt.dayofyear

print(f"RNA: {rna.shape[0]} samples x {rna.shape[1]} genes | IRIS: {anno['IRIS'].value_counts().to_dict()}")
# Your modelling starts below.


# 1. Preprocessing Before Modeling




## 1.1 Construct Day-of-Year

Seasonality is modeled over the annual cycle.

For each sample:

$
doy = \text{day of year from Date}
$

This variable must range from 1 to 365 (or 366).

Reason:
We model annual periodicity. The spline must be cyclic across the calendar year.



## 1.2 Convert SubjectID to Categorical

SubjectID must be treated as a categorical variable.

Reason:

Sampling is uneven.  
Different subjects appear at different times of year.

Without adjusting for subject baseline differences,
seasonality can be falsely detected due to composition imbalance.



## 1.3 Gene Filtering (Required)

Before genome-wide modeling:

1. Remove genes with excessive missing values  
2. Remove near-zero variance genes  
3. Remove extremely zero-inflated genes  

Reason:

Low-information genes produce unstable spline fits and inflate noise.



# 3. Mathematical Model

Let:

$
y_{it}^{(g)}
$

denote expression of gene $g$ for subject $i$ at time $t$.

We define three models.



## Model M0 (No Seasonality (Generalized Linear Mixed Model))

$
y_{it}^{(g)} = \alpha_i + \varepsilon_{it}
$

- $\alpha_i$ = subject-specific baseline  
- $\varepsilon_{it}$ = Gaussian error  

Tests whether gene varies only by subject baseline.



## Model M1 (Common Seasonality)

$
y_{it}^{(g)} = \alpha_i + f(doy_t) + \varepsilon_{it}
$

- $f(doy)$ = cyclic cubic spline  

Tests whether gene exhibits annual cyclic modulation.



## Model M2 (IRIS-Specific Seasonality)

$
y_{it}^{(g)}= \alpha_i + f(doy_t) + g(doy_t) \cdot IRIS_i + \varepsilon_{it}
$

- $g(doy)$ = deviation spline for IS  

Tests whether seasonal pattern differs between IR and IS.



# 4. Model Comparison

Use AIC to compare models:

$
\Delta AIC_{season} = AIC(M0) - AIC(M1)
$

$
\Delta AIC_{interaction} = AIC(M1) - AIC(M2)
$

Interpretation:

- Positive $\Delta AIC_{season}$: evidence for seasonality  
- Positive $\Delta AIC_{interaction}$: evidence that IR and IS differ in seasonal shape



# 5. Required Packages

You may use:

```python
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.gam.api import GLMGam
from statsmodels.gam.smooth_basis import CyclicCubicSplines
import matplotlib.pyplot as plt
```



# 6. Tasks

### Task 1 – Preprocessing
- Construct day_of_year
- Perform gene filtering
- Prepare SubjectID fixed effects

### Task 2 – Genome-wide Scan
- Fit M0 and M1 for all genes
- Compute $\Delta AIC_{season}$
- Rank genes

### Task 3 – IR vs IS Analysis
Either:
- Fit interaction model M2  
OR  
- Fit group-specific models separately

### Task 4 – Visualization
- Plot distribution of seasonal strength
- Plot seasonal curves for top genes
- Compare IR vs IS curves

### Task 5 – Interpretation
Explain:
- Is RNA globally seasonal?
- Is seasonality sparse?
